In [0]:
%python
display(
    dbutils.fs.ls("/Volumes/workspace/myra_invest/myra_data/bronze")
)

display(
    dbutils.fs.ls("/Volumes/workspace/myra_invest/myra_data/bronze/daily_stock_snapshot")
)

bronze_df = spark.read.format("delta").load("/Volumes/workspace/myra_invest/myra_data/bronze/daily_stock_snapshot")
display(bronze_df.limit(10))

print(bronze_df.count())
print(bronze_df.printSchema())

display(
    dbutils.fs.ls("dbfs:/Volumes/workspace/myra_invest/myra_data/bronze/daily_stock_snapshot")
)

(bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.myra_invest.bronze_stock_prices"))

In [0]:
DROP TABLE IF EXISTS workspace.myra_invest.bronze_stock_prices;

CREATE TABLE workspace.myra_invest.bronze_stock_prices
USING DELTA
LOCATION '/Volumes/workspace/myra_invest/myra_data/bronze/daily_stock_snapshot';

DROP TABLE IF EXISTS workspace.myra_invest.bronze_stock_prices;

CREATE TABLE workspace.myra_invest.bronze_stock_prices
USING DELTA
LOCATION 'dbfs:/Volumes/workspace/myra_invest/myra_data/bronze/daily_stock_snapshot';

In [0]:
SELECT COUNT(*) FROM workspace.myra_invest.bronze_stock_prices;

SELECT * FROM workspace.myra_invest.bronze_stock_prices LIMIT 10;

SELECT
    COUNT(*)                         AS total_records,
    COUNT(DISTINCT symbol)           AS total_symbols,
    MIN(tradeDate)                   AS earliest_trade_date,
    MAX(tradeDate)                   AS latest_trade_date,
    COUNT_IF(close IS NULL)          AS null_close_prices,
    COUNT_IF(volume IS NULL)         AS null_volume,
    COUNT_IF(volume <= 0)            AS invalid_volume
FROM workspace.myra_invest.bronze_stock_prices;

SELECT
    symbol,
    tradeDate,
    COUNT(*) AS duplicate_count
FROM workspace.myra_invest.bronze_stock_prices
GROUP BY symbol, tradeDate
HAVING COUNT(*) > 1;

SELECT
    symbol,
    COUNT(*) AS records
FROM workspace.myra_invest.bronze_stock_prices
GROUP BY symbol
ORDER BY records DESC;